In [ ]:
prefix_path = '../..'

import sys
sys.path.append(prefix_path)

import json
import pandas as pd
import numpy as np

In [ ]:
DATASET_PATH = f'{prefix_path}/data/phd/phd.json'
SAVE_PATH = f'{prefix_path}/data/phd/phd_base.json'

#### Construct Base PhD dataset

In [ ]:
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)

# Skip css data
df = df[df['ccs_description'].isna()]
# Convert NaN to null
df = df.replace({np.nan: None})

print(f'PhD dataset size: {len(df)}')
df.head()

In [ ]:
df = df.reset_index(drop=True)
df = df.assign(couple_id=df.index)

df_yes_question = df.assign(
    question=df['yes_question'],
    question_gt=1,
)[["couple_id", "task", "hitem", "subject", "gt", "question", "image_id", "question_gt"]]

df_no_question = df.assign(
    question=df['no_question'],
    question_gt=0,
)[["couple_id", "task", "hitem", "subject", "gt", "question", "image_id", "question_gt"]]

df_base = pd.concat([df_yes_question, df_no_question], ignore_index=True)
# Clean up
df_base = df_base.sample(frac=1, random_state=42).reset_index(drop=True)
df_base = df_base.reset_index().rename(columns={'index': 'id'})
df_base['id'] = df_base['id'].astype(str)
df_base['couple_id'] = df_base['couple_id'].astype(str)

print("Base PhD dataset size:", len(df_base))
df_base.head()

In [ ]:
dataset_base = df_base.to_dict(orient="records")

with open(SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(dataset_base, f, indent=2)